# Notebook 24: Hazard & Exposure Baseline (data half only)
Generated C2-data products; no registry/app/golden changes.

In [1]:
# See processed outputs:
print("data/processed/nri_wrc_county_hazard_summary.csv")
print("data/processed/asset_exposure_tags.json")
print("data/processed/snapshots/nri_wrc_baseline_v1.json")


## Credibility Gate
Credibility table: `data/processed/hazard_exposure_credibility_table.csv`. Sanity stats: `data/processed/hazard_county_sanity_stats.csv`.

## C2.1 — County hazard baseline and delta surface

This addendum resolves Wave 5 D1. It derives the historical baseline solely by algebraically inverting C1.6's published four-epoch interpolation doctrine, then computes `delta = projected_value - historical_baseline`. It performs no new sampling, modeling, or statistical fitting. The canonical baseline is the arithmetic mean of the independently back-derived SSP245 and SSP370 historical values; both derivations are retained in the output because their small difference is caused by already-rounded projected values.

In [ ]:
# C2.1 generator — writes data/processed/county_climate_baseline.json
import json
from collections import defaultdict
from pathlib import Path

SOURCE = Path('../data/processed/county_climate_projections.json')
OUTPUT = Path('../data/processed/county_climate_baseline.json')
EPOCHS = ('2030', '2040', '2050', '2065')
UNITS = {
    'annual_mean_temp_f': 'degF', 'annual_precip_total_in': 'in/year',
    'cdd': 'degree_days', 'days_gt_100f': 'days/year', 'days_gt_95f': 'days/year',
    'hdd': 'degree_days', 'high_fire_danger_days': 'days/year',
    'max_consecutive_dry_days': 'days', 'precip_99p_daily_in': 'in/year',
    'snotel_swe_baseline_in': 'in', 'snotel_swe_projected_in': 'in',
    'water_stress_index': 'index_0_to_1',
}

def historical_value(values):
    # Invert C1.6 mapping: 2030=h*.183333... + w2035*.816666...;
    # 2040=.7*w2035+.3*w2050; 2050=w2035/30+29*w2050/30;
    # 2065=w2050/30+29*w2065/30.
    e30, e40, e50, e65 = (values[e] for e in EPOCHS)
    w2035 = (e40 * (29/30) - .3 * e50) / (.7 * (29/30) - .3/30)
    w2050 = (e50 - w2035/30) / (29/30)
    return (e30 - (49/60) * w2035) / (11/60)

source_data = json.loads(SOURCE.read_text())
by_key = defaultdict(dict)
for record in source_data['records']:
    by_key[(record['geoid'], record['metric'], record['percentile'], record['lens'])][record['epoch']] = record

baselines, deltas = [], []
for geoid, metric, percentile in sorted({key[:3] for key in by_key}):
    lens_rows = {lens: by_key[(geoid, metric, percentile, lens)] for lens in ('ssp245', 'ssp370')}
    if any(set(rows) != set(EPOCHS) for rows in lens_rows.values()):
        raise ValueError(f'incomplete C1.6 epoch surface for {(geoid, metric, percentile)}')
    derived = {lens: historical_value({epoch: rows[epoch]['value'] for epoch in EPOCHS}) for lens, rows in lens_rows.items()}
    template = lens_rows['ssp245']['2030']
    baseline = {
        'geoid': geoid, 'county_name': template['county_name'], 'state': template['state'],
        'metric': metric, 'unit': UNITS[metric], 'lens': 'historical', 'scenario': 'historical',
        'epoch': 'historical', 'percentile': percentile,
        'value': sum(derived.values()) / len(derived), 'source': template['source'],
        'method': 'c2_1_epoch_doctrine_linear_system_back_derivation_then_lens_mean',
        'confidence': template['confidence'], 'downscaling_method': template.get('downscaling_method'),
        'lens_derivations': derived,
    }
    baselines.append(baseline)
    for lens, rows in lens_rows.items():
        for epoch, projection in rows.items():
            deltas.append({
                'geoid': geoid, 'county_name': projection['county_name'], 'state': projection['state'],
                'metric': metric, 'unit': UNITS[metric], 'lens': lens, 'scenario': projection['scenario'],
                'epoch': epoch, 'percentile': percentile,
                'value': projection['value'] - baseline['value'], 'source': projection['source'],
                'method': 'c2_1_arithmetic_projection_minus_historical_baseline',
                'confidence': projection['confidence'], 'downscaling_method': projection.get('downscaling_method'),
                'baseline_key': [geoid, metric, percentile], 'baseline_value': baseline['value'],
                'projected_value': projection['value'], 'baseline_attribution': {k: baseline[k] for k in ('lens', 'scenario', 'epoch', 'percentile', 'source', 'method', 'confidence', 'downscaling_method')},
            })

output = {
    'schema_version': 'C2.1.0', 'source_schema_version': source_data['schema_version'],
    'baseline_reference': {'epoch': 'historical', 'window': '1991-2020', 'derivation': 'C1.6 epoch_doctrine inversion; arithmetic mean of lens derivations'},
    'units': UNITS, 'metrics': sorted(UNITS), 'lenses': ['ssp245', 'ssp370'], 'epochs': list(EPOCHS),
    'baselines': baselines, 'deltas': deltas,
}
OUTPUT.write_text(json.dumps(output, indent=2) + '\n')
print(f'{len(baselines)} baselines and {len(deltas)} deltas written to {OUTPUT}')
